In [15]:
from __future__ import annotations

import numpy.typing as npt
import torch

from cs336_basics.bpe_tokenizer import BpeTokenizer
from cs336_basics.LanguageModel import LanguageModel
from cs336_basics.AdamW import AdamW
from cs336_basics.utils import run_get_batch, run_get_lr_cosine_schedule, run_cross_entropy, run_gradient_clipping, run_save_checkpoint, run_load_checkpoint



In [16]:
vocab_size = 10000
context_length = 256
d_model = 512
num_layers = 4
num_heads = 16
d_ff = 1344

batch_size=32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype=torch.bfloat16

rope_theta = 10000
betas = [0.9, 0.95]
weight_decay = 1


iterations = 10000

# learning rate hyperparam
max_lr = 1e-2
min_lr = 1e-5
warmup_iters = iterations * 0.2
cosine_cycle_iters = iterations * 0.9
max_l2_norm = 1


top_p_threshold = 0.8

In [17]:

# build train data from tokenizer
merges_filepath = 'output/TinyStoriesV2-GPT4-train-heap-merges.json'
vocab_filepath = 'output/TinyStoriesV2-GPT4-train-heap-vocab.json'

print('loading bpe tokenizer')
tokenizer = BpeTokenizer.from_files(vocab_filepath, merges_filepath, ['<|endoftext|>'])



loading bpe tokenizer


In [18]:

lm = LanguageModel(vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta, device=device, dtype=dtype)
optimizer = AdamW(lm.parameters(), betas, weight_decay, lr=1e-3, eps=1e-8)

print('loading model')
run_load_checkpoint('checkpoints/75000_iteration', lm, optimizer)

loading model


74999

In [51]:
def predict_next_token(lm, input_data, top_p_threshold, itr):
    logits = lm.forward(input_data, len(input_data))

    # print('output logits', logits)
    tlida = 0.01

    # softmax
    max_entity = torch.amax(logits, dim=-1, keepdim=True)
    softmax_score = torch.exp((logits - max_entity) / tlida) / torch.sum(torch.exp((logits - max_entity) / tlida), dim=-1, keepdim=True)

    last_token_probs = softmax_score[..., -1, :]   # (B, V)
    if itr == 0:
        print(logits)
        print(max_entity, softmax_score)
        print(last_token_probs)
    sorted_probs, sorted_indices = torch.sort(
        last_token_probs,
        dim=-1,
        descending=True
    )

    # 3. cumulative probability
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # 4. 找出超过 top_p 的 tokens
    remove_mask = cumulative_probs > top_p_threshold

    # 保留“第一个使 cumulative >= top_p 的 token”
    remove_mask[..., 1:] = remove_mask[..., :-1].clone()
    remove_mask[..., 0] = False

    # 5. 把 nucleus 外面的 probability 设为 0
    sorted_probs = sorted_probs.masked_fill(remove_mask, 0.0)

    # 6. renormalize
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

    # 7. 根据 probability 随机 sample
    sampled_position = torch.multinomial(
        sorted_probs,
        num_samples=1
    )

    # 8. sampled_position 是排序后的位置，
    #    要转换回原来的 vocab token id
    next_token_id = torch.gather(
        sorted_indices,
        dim=-1,
        index=sampled_position
    )
    # idx = torch.argmax(last_token_probs, dim=-1) # (B,)

    tokens = input_data + [next_token_id.item()]

    return tokens


TODO: In the future, add padding + attention mask

In [52]:
def generate_text(prompts, tokenizer, lm, top_p_threshold, max_context_length ):

    outputs = []
    for i, prompt in zip(torch.arange(len(prompts)), prompts):
        print(i.item())
        output_tokens = tokenizer.encode(prompt)

        i=0
        end_ids = tokenizer.encode('<|endoftext|>')
        
        input_data = output_tokens
        while i < max_context_length and input_data[-1] not in end_ids:

            output_tokens = predict_next_token(lm, input_data, top_p_threshold, i)
            input_data = output_tokens
            i = i + 1

        result = tokenizer.decode(output_tokens)
        outputs.append(result)
    return outputs

In [53]:
prompts = [
    # 1. Basic story continuation
    "Once upon a time, there was a little girl named Lily who",

    # 2. Simple cause and effect
    "Tom was very hungry, so he went to the kitchen and",

    # 3. Dialogue
    'Lucy looked at her mother and said, "Mom, can I',

    # 4. Problem → story development
    "One day, Ben lost his favorite toy. He looked everywhere, but",

    # 5. Slightly unusual situation / generalization
    "The little dog saw a big red balloon floating in the sky and"
]


# compare weight difference

In [54]:


print('loading 100k model')
itr = run_load_checkpoint('checkpoints/10000_iteration', lm, optimizer)
# after loading 10K
state_10k = {
    k: v.detach().clone()
    for k, v in lm.state_dict().items()
}

# load 75K
run_load_checkpoint(
    'checkpoints/75000_iteration',
    lm,
    optimizer
)

total_diff = 0

for name, param in lm.state_dict().items():
    diff = (param - state_10k[name]).abs().mean().item()
    total_diff += diff
    print(name, diff)

loading 100k model
token_embeddings.weight 5.888938903808594e-05
layers.0.ln1.weight 0.000293731689453125
layers.0.attn.q_proj.weight 2.193450927734375e-05
layers.0.attn.k_proj.weight 1.895427703857422e-05
layers.0.attn.v_proj.weight 8.58306884765625e-05
layers.0.attn.output_proj.weight 8.678436279296875e-05
layers.0.ln2.weight 1.239776611328125e-05
layers.0.ffn.w1.weight 7.152557373046875e-05
layers.0.ffn.w2.weight 7.104873657226562e-05
layers.0.ffn.w3.weight 7.104873657226562e-05
layers.1.ln1.weight 0.00017261505126953125
layers.1.attn.q_proj.weight 4.410743713378906e-05
layers.1.attn.k_proj.weight 5.125999450683594e-05
layers.1.attn.v_proj.weight 7.581710815429688e-05
layers.1.attn.output_proj.weight 8.344650268554688e-05
layers.1.ln2.weight 0.000202178955078125
layers.1.ffn.w1.weight 7.104873657226562e-05
layers.1.ffn.w2.weight 7.534027099609375e-05
layers.1.ffn.w3.weight 7.152557373046875e-05
layers.2.ln1.weight 0.0001678466796875
layers.2.attn.q_proj.weight 4.076957702636719e-05


In [55]:
print('loading 100k model')
itr = run_load_checkpoint('checkpoints/10000_iteration', lm, optimizer)
print(itr)
with torch.no_grad():
    lm.eval()
    lm_100k_outputs = generate_text(prompts[:1], tokenizer, lm, top_p_threshold, context_length )
    for name, param in lm.named_parameters():
        print(name, param.flatten()[:5])
        break

loading 100k model
9999
0
tensor([[-6.0625, -6.1250, -5.9375,  ..., -5.9688, -5.9375, -5.6875],
        [-6.4688, -6.4688, -6.4688,  ..., -6.7188, -6.0625, -6.0000],
        [-6.4375, -6.4375, -6.3750,  ..., -6.5312, -6.2812, -5.7188],
        ...,
        [-0.1152, -0.1396, -0.1455,  ..., -0.2695,  0.3457,  0.4023],
        [-7.6875, -7.7500, -7.6875,  ..., -7.1875, -6.5000, -7.6562],
        [-4.0000, -4.0938, -3.9531,  ..., -4.2812, -4.0625, -4.4062]],
       device='cuda:0', dtype=torch.bfloat16)
tensor([[12.5000],
        [12.8750],
        [15.1250],
        [12.7500],
        [12.8125],
        [14.6875],
        [14.7500],
        [ 8.7500],
        [ 9.7500],
        [14.5000],
        [12.6250],
        [14.5000],
        [13.7500]], device='cuda:0', dtype=torch.bfloat16) tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0.,

In [56]:
print('loading 750k model')
itr = run_load_checkpoint('checkpoints/75000_iteration', lm, optimizer)
print(itr)

with torch.no_grad():
    lm.eval()
    lm_750k_outputs = generate_text(prompts[:1], tokenizer, lm, top_p_threshold, context_length )
    for name, param in lm.named_parameters():
        print(name, param.flatten()[:5])
        break

loading 750k model
74999
0
tensor([[-6.0625, -6.1250, -5.9375,  ..., -6.0000, -5.9375, -5.6875],
        [-6.5000, -6.4688, -6.4688,  ..., -6.7188, -6.0625, -6.0000],
        [-6.5000, -6.5000, -6.4375,  ..., -6.5938, -6.3125, -5.7812],
        ...,
        [-0.1475, -0.1729, -0.1768,  ..., -0.2969,  0.3145,  0.3652],
        [-7.7188, -7.7812, -7.7188,  ..., -7.2188, -6.5312, -7.6875],
        [-4.0312, -4.0938, -3.9844,  ..., -4.2812, -4.0625, -4.4062]],
       device='cuda:0', dtype=torch.bfloat16)
tensor([[12.5000],
        [12.8125],
        [15.0000],
        [12.7500],
        [12.8125],
        [14.6875],
        [14.7500],
        [ 8.7500],
        [ 9.7500],
        [14.4375],
        [12.6250],
        [14.5000],
        [13.7500]], device='cuda:0', dtype=torch.bfloat16) tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0.

In [33]:
lm_7500_outputs

['Once upon a time, there was a little girl named Lily who loved to play outside. One day, she found a big, red ball in the yard. She was so happy and wanted to play with it.\nLily ran to her mom and said, "Mom, I found a big, red ball. Can I play with it?" Her mom smiled and said, "Yes, you can play with it, but be careful not to break it."\nLily played with the big, red ball all day. She threw it, kicked it, and rolled it. The ball went up, up, up into the sky. Lily was so happy to have her ball back. She played with it all day long, and she knew that she would always remember the fun she had with her mom.\n<|endoftext|>',
 'Tom was very hungry, so he went to the kitchen and found some food. He ate the food and felt very happy.\nTom\'s mom saw him and said, "Tom, you are so hungry! You can eat all the food!" Tom was surprised but happy. He ate the food and felt full. Then, he ate the food and felt full.\nTom\'s mom came into the kitchen and saw the food. She said, "Tom, you did a gre

In [31]:
lm_100k_outputs

['Once upon a time, there was a little girl named Lily who loved to play outside. One day, she found a big, red ball in the yard. She was very happy and wanted to play with it.\nLily ran to her mom and said, "Mom, I found a big, red ball. Can I play with it?" Her mom smiled and said, "Yes, you can play with it, but be careful not to break it."\nLily played with the big, red ball all day. She threw it, kicked it, and rolled it. The ball went up, up, up into the sky. Lily was so happy to have her ball back. She played with it all day long, and she knew that she would always remember the fun she had with her mom.\n<|endoftext|>',
 'Tom was very hungry, so he went to the kitchen and found some food. He ate the food and felt happy.\nTom\'s mom saw him and said, "Tom, you are so hungry! You can eat all the food!" Tom was surprised but happy. He ate the food and felt full. Then, he ate the food and felt full.\nTom\'s mom came into the kitchen and saw the food. She said, "Tom, you did a great 